# Geosteering AI — Sprint O0/O1: Benchmark JAX GPU Completo (8 cenários)

**Template:** `validate_jax_gpu_sprinto0_o1.ipynb` (cópia adaptada de `validate_jax_gpu_v240.ipynb`)
**Branch alvo:** `feat/sprint-o1-quick-wins` (deve estar **pushada** no remoto)
**Hardware:** Colab Pro+ T4 ou A100

## Propósito (diferente do `validate_sprint_o0_o1_gpu.ipynb`)

| Notebook | Cenários | Propósito |
|:---------|:---------|:----------|
| `validate_sprint_o0_o1_gpu.ipynb` | A, B, E (3) | Anti-regressão rápida (T1.6 gate) |
| **`validate_jax_gpu_sprinto0_o1.ipynb` (este)** | **A–H (8)** | **Quantificar ganhos por cenário pós Sprint O1** |

Este notebook reusa a infraestrutura do benchmark A1.6 (v240) — mede TODOS os 8 cenários
canônicos (A, B, C, D, E, F, G, H) na branch O0/O1 e compara contra:

1. **Baseline pré-O1 (jax_gpu_t4 ou jax_gpu_a100 em `perf_baseline.json`)** — medida em v2.43
   antes das otimizações O1. Razão `medido/baseline > 1.0` indica ganho.
2. **Numba T4 LOCAL** (medido via subprocess no mesmo Colab) — gate ≥1.5× preserva
   o critério Sprint A1.6.

## Quick Wins O1 verificados

| # | Otimização | Impacto esperado |
|:-:|:-----------|:-----------------|
| 1 | `_layer_at_depth_vec` (NumPy vetorizado) | Multi-TR / multi-pos |
| 2 | `lax.scan(unroll=K)` em propagação | Camadas profundas |
| 3 | XLA persistent compilation cache | Cold-start ↓ |
| 4 | XLA `latency_hiding_scheduler` | Launch-bound ↓ |
| 5 | `jax.config.jax_compilation_cache_dir` API | (apoio QW#3) |
| 6 | Hankel LRU cache process-wide | Multi-freq ↑ |
| 7 | Sync GPU→CPU deferido | Throughput global |
| 8 | `jax.tree.map` consolida HtoD | Triple-nested vmap (G/H) |
| 9 | `_sanitize_profile_batch` vetorizado | Batch grande |


In [1]:
# === L5: Configurações JAX OBRIGATÓRIAS antes de qualquer import ===
# Equivalente ao 'NUMBA_NUM_THREADS no PAI' (L5 do mapeamento Numba→JAX):
# jax_enable_x64 e jax_compilation_cache_dir devem ser definidos ANTES
# de 'import jax' — após o import, jax_enable_x64 fica imutável.
import os

os.environ["JAX_ENABLE_X64"] = "1"                              # float64 — paridade Fortran <1e-12 requer float64
os.environ["JAX_PLATFORMS"] = "cuda,cpu"                        # MED-4: explicit em vez de "" (compat JAX 0.4.30+)
os.environ["JAX_COMPILATION_CACHE_DIR"] = "/content/jax_cache"  # L8: cache XLA em SSD Colab (acelera Run 1 em re-execuções)

print("JAX env vars configurados (antes de qualquer import):")
print(f"  JAX_ENABLE_X64             = {os.environ['JAX_ENABLE_X64']}")
print(f"  JAX_PLATFORMS              = {os.environ['JAX_PLATFORMS']!r}  (CUDA preferido, CPU fallback)")
print(f"  JAX_COMPILATION_CACHE_DIR  = {os.environ['JAX_COMPILATION_CACHE_DIR']}")


JAX env vars configurados (antes de qualquer import):
  JAX_ENABLE_X64             = 1
  JAX_PLATFORMS              = 'cuda,cpu'  (CUDA preferido, CPU fallback)
  JAX_COMPILATION_CACHE_DIR  = /content/jax_cache


## § 1 — Configurações da Sprint


In [ ]:
# ─── Configuração Sprint O0/O1 ───────────────────────────────────────────────
# Branch alvo (não tag — Sprint O0/O1 ainda não tem versão atribuída;
# ADR-0001 R2: versões vX.Y atribuídas no primeiro commit da sprint)
GIT_BRANCH       = "feat/sprint-o1-quick-wins"
GIT_REPO_URL     = "https://github.com/daniel-guitarplayer-8/geosteering-ai.git"
REPO_DIR         = "/content/geosteering-ai"
OUTPUT_DRIVE_DIR = "/content/drive/MyDrive/Geosteering_AI/sprint_o0_o1/"

PARIDADE_TOL     = 1e-12   # gate de paridade rigoroso (inviolável)
N_RUNS_HOT       = 4       # runs hot-cache (Runs 2-5 após Run 1 cold)
N_MODELS         = 50      # modelos geológicos por run (5 camadas, semente 42)
SEED             = 42      # reprodutibilidade — idêntico ao CLI _build_models

# Workers/threads pinados para o CLI Numba baseline (MED-5).
NUMBA_BASELINE_WORKERS = 4
NUMBA_BASELINE_THREADS = 1

# Diretório de cache JAX (L8): deve existir antes de import jax
os.makedirs("/content/jax_cache", exist_ok=True)

# --- Cenários canônicos (espelha geosteering_ai/cli/benchmark.py:113-147) ---
SCENARIOS_JAX = {
    "A": {
        "n_pos": 1, "freqs": (20000.0,), "trs": (1.0,), "dips_jax": (0.0,),
        "numba_baseline_historical_mod_h": 1_180_000,
    },
    "B": {
        "n_pos": 100, "freqs": (20000.0,), "trs": (1.0,), "dips_jax": (0.0,),
        "numba_baseline_historical_mod_h": 320_000,
    },
    "C": {
        "n_pos": 100,
        "freqs": (2000.0, 20000.0, 100000.0, 400000.0),
        "trs": (1.0,), "dips_jax": (0.0,),
        "numba_baseline_historical_mod_h": 119_000,
    },
    "D": {
        "n_pos": 1, "freqs": (20000.0,),
        "trs": (0.5, 1.0, 1.5, 2.0), "dips_jax": (0.0,),
        "numba_baseline_historical_mod_h": None,
    },
    "E": {
        "n_pos": 600, "freqs": (20000.0,), "trs": (1.0,), "dips_jax": (0.0,),
        "numba_baseline_historical_mod_h": 122_000,
    },
    "F": {
        "n_pos": 100,
        "freqs": (2000.0, 20000.0, 100000.0, 400000.0),
        "trs": (0.5, 1.0, 1.5, 2.0), "dips_jax": (0.0,),
        "numba_baseline_historical_mod_h": None,
    },
    "G": {
        "n_pos": 100,
        "freqs": (2000.0, 20000.0, 100000.0, 400000.0),
        "trs": (0.5, 1.0, 1.5, 2.0),
        "dips_jax": (0.0, 15.0, 30.0, 45.0),
        "numba_baseline_historical_mod_h": 45_000,
    },
    "H": {
        # JAX validador cap dip em 89°. H NÃO entra no gate de paridade exato.
        "n_pos": 100,
        "freqs": (1e3, 2e3, 5e3, 1e4, 2e4, 5e4, 1e5, 2e5),
        "trs": (0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.5),
        "dips_jax": (0.0, 12.5, 25.0, 37.5, 50.0, 62.5, 75.0, 87.5),
        "numba_baseline_historical_mod_h": None,
    },
}

# Modelos por cenário calibrados para T4 (15 GB VRAM).
N_MODELS_PER_SCENARIO = {
    "A": N_MODELS, "B": N_MODELS, "C": N_MODELS, "D": N_MODELS,
    "E": N_MODELS, "F": 20, "G": 5, "H": 5,
}

# Timeout do subprocess CLI por cenário.
NUMBA_CLI_TIMEOUT_S = {scen: (1800 if scen == "H" else 600) for scen in SCENARIOS_JAX}

# Quick Wins O1 — Sprint O1 thresholds esperados (ganho mínimo por cenário)
# Baseado em qual QW atinge cada cenário. Valor 1.00 = neutro (sem regressão).
# Estes são EXPECTATIVAS para análise — não bloqueiam o gate principal (≥1.5× Numba).
O1_GAIN_EXPECTATIONS = {
    "A": 1.00,   # overhead-bound — QW#7 marginal
    "B": 1.05,   # bandwidth — QW#1 + QW#7 ajudam pouco
    "C": 1.20,   # multi-freq → QW#6 (Hankel LRU) deveria entregar +20%
    "D": 1.15,   # multi-TR → QW#1 vetorizado ajuda
    "E": 1.05,   # n_pos=600 com fori_loop — QW#2 (unroll) parcial
    "F": 1.15,   # multi-freq+TR → QW#6 + QW#1
    "G": 1.25,   # triple-nested → QW#8 HtoD consolidation entrega mais
    "H": 1.00,   # stress test — ganhos opcionais
}

print(f"GIT_BRANCH={GIT_BRANCH!r} | N_MODELS={N_MODELS} | N_RUNS_HOT={N_RUNS_HOT} (+1 cold) | tol={PARIDADE_TOL:.0e}")
print(f"Numba CLI baseline: workers={NUMBA_BASELINE_WORKERS} threads={NUMBA_BASELINE_THREADS}")
print(f"Cenários: {list(SCENARIOS_JAX.keys())}")
print(f"N_MODELS por cenário: {N_MODELS_PER_SCENARIO}")
print(f"Expectativas ganho O1 vs pré-O1: {O1_GAIN_EXPECTATIONS}")


## § 2 — Setup do Ambiente

Clona o repositório, instala dependências e valida que a GPU está
disponível com float64. **H1 fix**: detecção via `jax.default_backend()`
(compatível com JAX 0.4.14+) em vez do deprecado `device.platform == "gpu"`.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import subprocess, sys, shutil

# Clone — fail-fast se branch não estiver no remoto (sem fallback silencioso)
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

print(f"Clonando branch {GIT_BRANCH!r} do remoto ...")
_clone = subprocess.run(
    ["git", "clone", "--depth=1", "--branch", GIT_BRANCH, GIT_REPO_URL, REPO_DIR],
    capture_output=True, text=True,
)
if _clone.returncode != 0:
    raise RuntimeError(
        f"FALHA AO CLONAR branch {GIT_BRANCH!r}.\n"
        f"stderr: {_clone.stderr.strip()}\n\n"
        f"Verifique: git push -u origin {GIT_BRANCH}"
    )

%cd /content/geosteering-ai

# ── Instalar geosteering_ai[all] (sem [dev] → sem pytest-qt) ──────────────────
print("Instalando geosteering_ai[all] ...")
_install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[all]"],
    capture_output=True, text=True,
)
if _install.returncode != 0:
    print(_install.stdout[-2000:])
    print(_install.stderr[-2000:])
    raise RuntimeError("pip install -e .[all] FALHOU")

# ── JAX CUDA explícito — TF pode instalar jaxlib CPU como transitividade ────
print("Garantindo JAX CUDA (jax[cuda12]) ...")
_jax_install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "jax[cuda12]"],
    capture_output=True, text=True,
)
if _jax_install.returncode != 0:
    print(f"  [WARN] jax[cuda12] retornou {_jax_install.returncode}: {_jax_install.stderr.strip()[:200]}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "pytest>=7", "pytest-json-report>=1.5"],
    check=True,
)
print("✓ Dependências instaladas")

# Imports — apenas APÓS env vars (Célula 1) E APÓS pip installs
import json
import datetime
import re
import statistics
import time
from pathlib import Path
from typing import Optional

# ── Diagnóstico nvidia-smi ANTES de import jax (distingue caso A vs B) ─────
_nvs = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
     "--format=csv,noheader"],
    capture_output=True, text=True,
)
_gpu_hw_available = _nvs.returncode == 0 and bool(_nvs.stdout.strip())
if _gpu_hw_available:
    print(f"\n[nvidia-smi] {_nvs.stdout.strip()}")
else:
    print(f"\n[nvidia-smi] Nenhuma GPU detectada — runtime CPU?")

import jax
import jax.numpy as jnp
import numpy as np

# Validar GPU
_BACKEND = jax.default_backend()
_DEVICES = jax.devices()
_GPU_DEVS = [d for d in _DEVICES if "gpu" in str(d).lower() or "cuda" in str(d).lower()]

IS_GPU = _BACKEND in ("gpu", "cuda") and bool(_GPU_DEVS)
if not IS_GPU:
    _diag = [
        f"GPU NÃO detectada pelo JAX.",
        f"  backend={_BACKEND!r}  devices={_DEVICES}  jax={jax.__version__}",
        f"  nvidia-smi GPU: {_gpu_hw_available}",
    ]
    if _gpu_hw_available:
        _diag += [
            "  Hardware GPU presente, mas JAX sem CUDA.",
            "  ► Solução: Runtime → Restart runtime → Execute desde Célula 1",
        ]
    else:
        _diag += ["  ► Solução: Runtime → Change runtime type → T4 GPU (ou A100)"]
    raise AssertionError("\n".join(_diag))

print(f"Backend default: {_BACKEND!r} ✓")
print(f"Dispositivos GPU: {_GPU_DEVS}")

assert jax.config.jax_enable_x64, "JAX_ENABLE_X64 deve ser setado ANTES de import jax (Célula 1)."
print(f"float64 habilitado: {jax.config.jax_enable_x64} ✓")

# Hardware label — para selecionar baseline correta em §7
_gpu_name_raw = _nvs.stdout.strip()
_gpu_name_lower = _gpu_name_raw.lower()
RUNTIME_LABEL = (
    "a100" if "a100" in _gpu_name_lower
    else ("t4" if "t4" in _gpu_name_lower else "gpu_unknown")
)
print(f"\n[GPU] {_gpu_name_raw}")
print(f"RUNTIME_LABEL = {RUNTIME_LABEL!r}")

# Pin commit hash
try:
    GIT_COMMIT = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=REPO_DIR
    ).decode().strip()
except (subprocess.CalledProcessError, FileNotFoundError):
    GIT_COMMIT = f"unknown-{GIT_BRANCH}"
print(f"Commit: {GIT_COMMIT[:12]} (branch={GIT_BRANCH})")


## § 3 — Warmup por-cenário com Batched API (C1 + H3 fixes)

**Por que warmup por-cenário com batch size exato é crítico:**

A v2.40 usava um warmup global com shape fixo `(nf=2, nTR=3, nAng=2)` que
**nenhum cenário utilizava** — todo Run 1 de cada cenário recompilava JIT
XLA (35-119× cold-start em B/C/E). Pior ainda, o warmup chamava
`simulate_multi_jax(models[0], ...)` em loop Python, e o `_BUCKET_JIT_CACHE`
(chaveado em `(camad_t, camad_r, n, npt)`) acumulava ~50 buckets distintos
durante Run 1 (1 por modelo aleatório).

**Solução A1.5 (v2.42) + A1.6 (warmup com batch correto):**
- `simulate_multi_jax_batched` usa `_UNIFIED_JIT_CACHE` keyed `(n, npt)` —
  invariante a valores de modelo (resolve bucket explosion / H3).
- **Mas o XLA cache** (subjacente ao `jax.jit` retornado) ainda chaveia
  no **shape completo dos args**, incluindo o `n_models` (axis size da
  outer `jax.vmap`). Warmup com batch=2 NÃO cobre benchmark com batch=50.
- **CRIT-2 fix**: warmup usa `np.tile(template, (N_MODELS_PER_SCENARIO[s], 1))`
  por cenário — batch exato do benchmark.

Lição L4 do simulador Numba aplicada: aquecer com dados anisotrópicos
(ρ_v ≠ ρ_h) + dip ≠ 0° para acionar todos os paths do código.


In [4]:
from geosteering_ai.simulation import simulate_multi_jax_batched

# Modelo anisotrópico com dip ≠ 0° (cobre paths JIT que isotropico+dip=0 não acionam — L4)
_RHO_H_WARM_TEMPLATE = np.array([2.0, 15.0, 120.0, 8.0, 40.0], dtype=np.float64)
_RHO_V_WARM_TEMPLATE = _RHO_H_WARM_TEMPLATE * np.array([1.8, 2.5, 3.0, 1.5, 2.0], dtype=np.float64)
_ESP_WARM_TEMPLATE   = np.array([6.0, 9.0, 7.0], dtype=np.float64)   # 5 camadas → 3 espessuras internas

print("=" * 75)
print("WARMUP por-cenário com batch size EXATO (XLA cache key = shape completa)")
print("=" * 75)

for scen_name, scen in SCENARIOS_JAX.items():
    n_models_scen = N_MODELS_PER_SCENARIO[scen_name]
    n_pos = scen["n_pos"]
    n_combos = len(scen["freqs"]) * len(scen["trs"]) * len(scen["dips_jax"])

    # CRIT-2 fix: tile template para batch EXATO do benchmark
    rho_h_warm = np.tile(_RHO_H_WARM_TEMPLATE, (n_models_scen, 1))  # (n_models_scen, 5)
    rho_v_warm = np.tile(_RHO_V_WARM_TEMPLATE, (n_models_scen, 1))  # (n_models_scen, 5)
    esp_warm   = np.tile(_ESP_WARM_TEMPLATE,   (n_models_scen, 1))  # (n_models_scen, 3)

    pos = np.linspace(-5.0, 5.0, n_pos).astype(np.float64)  # match CLI benchmark.py:259

    t0 = time.perf_counter()
    try:
        res = simulate_multi_jax_batched(
            rho_h_warm, rho_v_warm, esp_warm, pos,
            frequencies_hz=list(scen["freqs"]),
            tr_spacings_m=list(scen["trs"]),
            dip_degs=list(scen["dips_jax"]),
        )
        _ = res.H_tensor.shape  # H2 defensivo
        print(f"  {scen_name}: n_models={n_models_scen:>2d}, n_pos={n_pos:>3d}, combos={n_combos:>3d} (nf={len(scen['freqs'])}, nTR={len(scen['trs'])}, nAng={len(scen['dips_jax'])}): {time.perf_counter()-t0:6.2f}s")
    except Exception as _exc:
        if "RESOURCE_EXHAUSTED" in str(_exc) or "Out of memory" in str(_exc):
            print(f"  {scen_name}: OOM com n_models={n_models_scen} — warmup ignorado; Run 1 será cold-start")
        else:
            raise

print("\n✓ Warmup concluído. XLA cache populado para batch exato de cada cenário.")
print("   Run 1 do benchmark deve estar HOT — se cold/hot ratio > 2×, warmup falhou.")


WARMUP por-cenário com batch size EXATO (XLA cache key = shape completa)
  A: n_models=50, n_pos=  1, combos=  1 (nf=1, nTR=1, nAng=1):  16.23s
  B: n_models=50, n_pos=100, combos=  1 (nf=1, nTR=1, nAng=1):  21.85s
  C: n_models=50, n_pos=100, combos=  4 (nf=4, nTR=1, nAng=1):  23.35s
  D: n_models=50, n_pos=  1, combos=  4 (nf=1, nTR=4, nAng=1):  13.12s
  E: n_models=50, n_pos=600, combos=  1 (nf=1, nTR=1, nAng=1):  25.75s
  F: n_models=20, n_pos=100, combos= 16 (nf=4, nTR=4, nAng=1):  23.35s
  G: n_models= 5, n_pos=100, combos= 64 (nf=4, nTR=4, nAng=4):  23.78s
  H: OOM com n_models=5 — warmup ignorado; Run 1 será cold-start

✓ Warmup concluído. XLA cache populado para batch exato de cada cenário.
   Run 1 do benchmark deve estar HOT — se cold/hot ratio > 2×, warmup falhou.


## § 3.5 — Verificação dos Quick Wins Sprint O1 (9 checks)

Confirma que os 9 Quick Wins estão ativos e configurados corretamente
**antes** de rodar o benchmark completo. Os checks são informativos —
qualquer FAIL aqui registra no JSON final mas não bloqueia o benchmark.
Um FAIL aqui indica que a branch O1 não foi clonada corretamente ou
que uma otimização foi revertida.


In [ ]:
# Quick Wins #1–#9 — verificação ativa (espelha validate_sprint_o0_o1_gpu.ipynb §3)
qw_checks = []

# ── #3 XLA persistent cache dir ───────────────────────────────────────────────
_cache_dir = os.environ.get("JAX_COMPILATION_CACHE_DIR", "")
qw_checks.append(("#3 XLA cache dir setado", bool(_cache_dir), _cache_dir or "(não setado)"))

# ── #4 XLA flag latency_hiding_scheduler ──────────────────────────────────────
_xla = os.environ.get("XLA_FLAGS", "")
qw_checks.append(
    ("#4 XLA flag latency_hiding_scheduler",
     "xla_gpu_enable_latency_hiding_scheduler=true" in _xla,
     _xla[:120] or "(não setado)")
)

# ── #4 VRAM allocator config ──────────────────────────────────────────────────
qw_checks.append(
    ("#4 XLA_PYTHON_CLIENT_PREALLOCATE=false",
     os.environ.get("XLA_PYTHON_CLIENT_PREALLOCATE") == "false",
     os.environ.get("XLA_PYTHON_CLIENT_PREALLOCATE", "(não setado)"))
)

# ── #5 jax.config.jax_compilation_cache_dir ──────────────────────────────────
_jax_cache_cfg = getattr(jax.config, "jax_compilation_cache_dir", None) or ""
qw_checks.append(
    ("#5 jax.config.jax_compilation_cache_dir (env propagado)",
     bool(_cache_dir),
     _jax_cache_cfg or f"(env={_cache_dir!r})")
)

# ── #2 SimulationConfig.unroll_layer_loop default ─────────────────────────────
try:
    from geosteering_ai.simulation.config import SimulationConfig
    _cfg = SimulationConfig()
    _unroll = getattr(_cfg, "unroll_layer_loop", None)
    qw_checks.append(
        ("#2 SimulationConfig.unroll_layer_loop default=True",
         _unroll is True,
         f"unroll_layer_loop={_unroll!r}")
    )
except Exception as _e:
    qw_checks.append(("#2 unroll_layer_loop", False, f"import erro: {_e}"))

# ── #6 Hankel LRU cache process-wide ──────────────────────────────────────────
try:
    from geosteering_ai.simulation._jax.kernel import get_hankel_filter_jax
    _t0 = time.perf_counter()
    _kr1, _w0_1, _w1_1 = get_hankel_filter_jax("werthmuller_201pt")
    _kr1.block_until_ready()
    _t1 = time.perf_counter()
    _kr2, _w0_2, _w1_2 = get_hankel_filter_jax("werthmuller_201pt")
    _kr2.block_until_ready()
    _t2 = time.perf_counter()
    _first_ms, _second_ms = (_t1-_t0)*1000, (_t2-_t1)*1000
    _speedup = _first_ms / max(_second_ms, 1e-6)
    qw_checks.append(
        ("#6 Hankel LRU cache speedup (2ª chamada)",
         _speedup > 10,
         f"1ª={_first_ms:.1f}ms → 2ª={_second_ms:.3f}ms ({_speedup:.0f}× speedup)")
    )
except Exception as _e:
    qw_checks.append(("#6 Hankel LRU cache", False, f"erro: {_e}"))

# ── #1 _layer_at_depth_vec (NumPy vetorizado) ─────────────────────────────────
try:
    from geosteering_ai.simulation._jax.multi_forward import _layer_at_depth_vec
    _z = np.array([-1.0, 0.5, 1.5, 100.0])
    _prof = np.array([0.0, 1.0, 2.0])
    _idx = _layer_at_depth_vec(3, _z, _prof)
    _expected = [0, 0, 1, 2]
    _ok = _idx.shape == (4,) and _idx.tolist() == _expected
    qw_checks.append(
        ("#1 _layer_at_depth_vec funcional",
         _ok, f"idx={_idx.tolist()} (esperado {_expected})")
    )
except Exception as _e:
    qw_checks.append(("#1 _layer_at_depth_vec", False, f"erro: {_e}"))

# ── #9 _sanitize_profile_batch importável ─────────────────────────────────────
try:
    from geosteering_ai.simulation._jax.multi_forward import _sanitize_profile_batch
    qw_checks.append(("#9 _sanitize_profile_batch importável", True, "OK"))
except Exception as _e:
    qw_checks.append(("#9 _sanitize_profile_batch", False, f"erro: {_e}"))

# ── Relatório ─────────────────────────────────────────────────────────────────
print("=" * 75)
print("Sprint O1 — Verificação de Quick Wins (9 checks)")
print("=" * 75)
_qw_n_pass = 0
for label, ok, detail in qw_checks:
    status = "✓ PASS" if ok else "✗ FAIL"
    if ok:
        _qw_n_pass += 1
    print(f"  {status} {label}")
    if not ok or "speedup" in label.lower():
        print(f"         {detail}")

print("=" * 75)
QW_ALL_PASS = _qw_n_pass == len(qw_checks)
print(f"  {_qw_n_pass}/{len(qw_checks)} Quick Wins confirmados {'✓' if QW_ALL_PASS else '⚠'}")
if not QW_ALL_PASS:
    print(f"  ⚠ Alguns QW falharam — investigar antes de continuar")


## § 4 — Paridade Fortran < 1e-12 (testes pytest -m gpu)

Os testes `@pytest.mark.gpu` validam paridade JAX GPU vs Fortran < 1e-12.
Com `JAX_PLATFORMS="cuda,cpu"` (Célula 1), o conftest detecta GPU
e executa os testes que seriam SKIPPED em CPU.


In [5]:
# Executa testes JAX GPU com relatório JSON
!python -m pytest tests/test_simulation_jax_*.py \
    -m gpu \
    -v \
    --tb=short \
    --json-report \
    --json-report-file=/tmp/jax_gpu_pytest.json \
    2>&1 | tail -40

# Carregar relatório pytest UMA VEZ — reutilizado em § 8 e § 9 (MED-3 fix)
with open("/tmp/jax_gpu_pytest.json") as _f:
    _pytest_report = json.load(_f)
_summary = _pytest_report["summary"]
print(f"\nResumo pytest: {_summary}")
assert _summary.get("failed", 0) == 0, (
    f"GATE PARIDADE FALHOU: {_summary['failed']} testes falharam. "
    "Investigar antes de prosseguir com benchmark."
)
print(f"✓ Paridade: {_summary.get('passed', 0)} PASS / 0 FAIL")


tests/test_simulation_jax_sprint12.py::test_vmap_real_bucketed_strategy_also_works PASSED [ 78%]
tests/test_simulation_jax_sprint12.py::test_chunk_size_config_validation PASSED [ 79%]
tests/test_simulation_jax_sprint12.py::test_chunked_unified_parity_100pos PASSED [ 79%]
tests/test_simulation_jax_sprint12.py::test_chunked_unified_parity_600pos_3freqs PASSED [ 80%]
tests/test_simulation_jax_sprint12.py::test_chunked_non_divisible_padding PASSED [ 81%]
tests/test_simulation_jax_sprint12.py::test_chunked_via_simulate_multi_jax PASSED [ 81%]
tests/test_simulation_jax_sprint13.py::TestSprint143JITCacheInfo::test_jit_cache_info_empty PASSED [ 82%]
tests/test_simulation_jax_sprint13.py::TestSprint143JITCacheInfo::test_jit_cache_info_after_simulate_unified PASSED [ 82%]
tests/test_simulation_jax_sprint13.py::TestSprint143JITCacheInfo::test_jit_cache_info_vram_estimate PASSED [ 83%]
tests/test_simulation_jax_sprint13.py::TestSprint143JITCacheInfo::test_jit_cache_info_idempotent_readonly PASSED 

In [6]:
# Sanity check da API batched (shape, dtype, ausência de NaN/Inf)
_n_models_check = 3
_rho_h_check = np.tile(np.array([1.0, 10.0, 100.0, 10.0, 1.0]), (_n_models_check, 1))
_rho_v_check = _rho_h_check * 2.0  # TIV anisotrópico
_esp_check   = np.tile(np.array([5.0, 10.0, 5.0]), (_n_models_check, 1))
_pos_check   = np.linspace(-5.0, 5.0, 50).astype(np.float64)

_res = simulate_multi_jax_batched(
    _rho_h_check, _rho_v_check, _esp_check, _pos_check,
    frequencies_hz=[20000.0], tr_spacings_m=[1.0], dip_degs=[0.0],
)
_ = _res.H_tensor.shape  # H2 defensivo

assert _res.H_tensor.dtype == np.complex128,                    f"dtype errado: {_res.H_tensor.dtype}"
assert _res.H_tensor.shape == (_n_models_check, 1, 1, 50, 1, 9), f"shape errado: {_res.H_tensor.shape}"
assert not np.any(np.isnan(_res.H_tensor.real)),                 "NaN em H_tensor.real!"
assert not np.any(np.isnan(_res.H_tensor.imag)),                 "NaN em H_tensor.imag!"
assert not np.any(np.isinf(_res.H_tensor)),                      "Inf em H_tensor!"

print(f"Shape:  {_res.H_tensor.shape}  ✓  (n_models={_n_models_check})")
print(f"dtype:  {_res.H_tensor.dtype}  ✓")
print(f"NaN:    ausente  ✓")
print(f"Inf:    ausente  ✓")
print(f"|Hzz| médio (modelo 0): {np.abs(_res.H_tensor[0,0,0,:,0,8]).mean():.6e}  (componente axial ZZ)")


Shape:  (3, 1, 1, 50, 1, 9)  ✓  (n_models=3)
dtype:  complex128  ✓
NaN:    ausente  ✓
Inf:    ausente  ✓
|Hzz| médio (modelo 0): 1.582982e-01  (componente axial ZZ)


## § 5 — Baseline Numba CPU no MESMO hardware T4 (M3 fix)

A auditoria v2.40.4 identificou que o gate ≥1.5× Numba era inválido
porque o baseline estava medido em hardware diferente (Intel i9-9980HK
8C/16T Mac Intel) — Colab T4 usa `n1-standard-4` (4 vCPUs Xeon, sem HT),
~50-60% do throughput Numba do Mac Intel.

**M3 fix**: mede o baseline Numba **localmente no T4** via subprocess
`geosteering-cli benchmark --scenario X --n N --workers 4 --threads 1`.
Workers/threads pinados (MED-5 fix) para reprodutibilidade — sem isto, o
auto-detect `recommend_default_parallelism` heuristic pode variar entre
sessões Colab. O CLI usa `simulate_multi` (Numba ProcessPool) idêntico
ao código de produção. Output formato `"Cenário X — N,NNN,NNN mod/h"` é
parseado via regex robusto.

**Cenário H — nota M1**: o validador JAX limita `dip_degs ≤ 89°`
(ver `multi_forward.py:880`), enquanto o CLI Numba usa 90°. Os
throughputs **não são exatamente** apples-to-apples em H — diferença
documentada e gate só avaliado em A/B/E.

**Cenários D/F/H**: baselines podem retornar `None` se o CLI subprocess
falhar (timeout/erro). Esses cenários ficam fora do gate por padrão.
Cenário H tem timeout estendido (1800s — HIGH-2 fix).


In [7]:
def measure_numba_baseline_t4(
    scen_name: str,
    n_models: int,
    timeout_s: int,
    workers: int = NUMBA_BASELINE_WORKERS,
    threads: int = NUMBA_BASELINE_THREADS,
) -> Optional[float]:
    """Mede baseline Numba CPU no MESMO hardware T4 via CLI subprocess (M3 fix).

    Invoca ``geosteering-cli benchmark --scenario X --n N --workers W --threads T``
    e parseia o throughput em mod/h. Usa os cenários canônicos do CLI (A-H)
    — não passa --frequencies/--dips/--tr-spacings (mantém defaults).

    Args:
        scen_name: identificador do cenário (A..H, deve existir no CLI)
        n_models: número de modelos
        timeout_s: timeout do subprocess em segundos
        workers: --workers (default 4, pinado MED-5)
        threads: --threads (default 1, pinado MED-5)

    Returns:
        Throughput em mod/h (float) ou None em caso de erro/timeout/CLI ausente.
    """
    cmd = [
        "geosteering-cli", "benchmark",
        "--scenario", scen_name,
        "--n", str(n_models),
        "--workers", str(workers),
        "--threads", str(threads),
    ]
    try:
        result = subprocess.run(
            cmd, capture_output=True, text=True, check=True, timeout=timeout_s,
        )
    except subprocess.CalledProcessError as exc:
        print(f"    [WARN] CLI exit {exc.returncode}: {(exc.stderr or '')[:200]}")
        return None
    except subprocess.TimeoutExpired:
        print(f"    [WARN] CLI timeout (>{timeout_s}s)")
        return None
    except (FileNotFoundError, OSError) as exc:
        # HIGH-1 fix: geosteering-cli não em PATH ou erro de OS
        print(f"    [WARN] CLI não encontrado/erro OS: {exc}")
        return None

    # Parse "Cenário X — 1,180,000 mod/h" (benchmark.py:319)
    match = re.search(r"—\s+([\d,]+)\s+mod/h", result.stdout)
    if not match:
        print(f"    [WARN] Parse falhou em stdout: {result.stdout[:200]!r}")
        return None
    return float(match.group(1).replace(",", ""))


print("=" * 75)
print(f"MEDINDO BASELINE NUMBA CPU NO T4 LOCAL  (workers={NUMBA_BASELINE_WORKERS}, threads={NUMBA_BASELINE_THREADS})")
print("=" * 75)

numba_t4_baselines = {}
for scen_name in SCENARIOS_JAX:
    n_m = N_MODELS_PER_SCENARIO[scen_name]
    timeout = NUMBA_CLI_TIMEOUT_S[scen_name]
    print(f"\n  Cenário {scen_name} (n={n_m}, timeout={timeout}s)...", end=" ", flush=True)
    t0 = time.perf_counter()
    th = measure_numba_baseline_t4(scen_name, n_m, timeout_s=timeout)
    elapsed = time.perf_counter() - t0
    numba_t4_baselines[scen_name] = th
    if th is not None:
        print(f"{th:>12,.0f} mod/h  ({elapsed:.1f}s)")
    else:
        print(f"FALHA ({elapsed:.1f}s) — gate cenário {scen_name} indisponível")

print("\n✓ Baseline Numba T4 LOCAL medido.")


MEDINDO BASELINE NUMBA CPU NO T4 LOCAL  (workers=4, threads=1)

  Cenário A (n=50, timeout=600s)...    2,692,990 mod/h  (93.7s)

  Cenário B (n=50, timeout=600s)...       90,158 mod/h  (181.9s)

  Cenário C (n=50, timeout=600s)...       26,509 mod/h  (18.8s)

  Cenário D (n=50, timeout=600s)...      964,837 mod/h  (16.0s)

  Cenário E (n=50, timeout=600s)...       22,653 mod/h  (20.2s)

  Cenário F (n=20, timeout=600s)...        8,079 mod/h  (187.3s)

  Cenário G (n=5, timeout=600s)...        2,093 mod/h  (23.8s)

  Cenário H (n=5, timeout=1800s)...          416 mod/h  (78.3s)

✓ Baseline Numba T4 LOCAL medido.


## § 6 — Benchmark JAX batched (C1 + C2 + H2 + M2 fixes)

Mede throughput em mod/h da API `simulate_multi_jax_batched` (Sprint A1.5,
v2.42) que aplica `jax.vmap` sobre o eixo `n_models` — substitui o loop
Python `for m in models: simulate_multi_jax(...)` por **1 trace XLA + 1
sync GPU→CPU** para todo o batch.

**C2 fix**: Run 1 (cold-start, pode triggerar compile XLA se warmup não
cobriu) é reportado SEPARADO de Runs 2-5 (hot-cache). O gate usa
`statistics.median(runs_2_to_5)`, NÃO `statistics.median([cold] + hot)`.

**H2 fix**: leitura de `result.H_tensor.shape` após cada call força
materialização (defensivo — a função já chama `block_until_ready()` +
`np.asarray()` internamente em `multi_forward.py:1111-1112`).

**M2 fix (design)**: `_UNIFIED_JIT_CACHE` chaveado `(n, npt)` é invariante
a `esp_batch` — variância de espessuras NÃO causa recompile. RNG mantido
para reprodutibilidade com CLI.

**Métricas reportadas:**
- `ratio_cold_to_hot`: cold/hot. ≈ 1.0 indica warmup efetivo; > 2× indica
  cold-start residual (warmup falhou ou XLA cache evicted).
- `latency_ms_per_model_hot`: `3.6e6 / median_hot` ms — tempo por modelo
  (CRIT-1 fix: fórmula correta sem multiplicar por n_models).


In [8]:
def _build_models(n_models: int, seed: int = 42) -> list:
    """Gera modelos 5-camada determinísticos (idêntico ao CLI _build_models).

    Mantém paridade EXATA com geosteering_ai/cli/benchmark.py:150-184
    (sentinel: rng.uniform(1.0, 100.0, size=5) para rho_h, * uniform(1, 3)
    para rho_v, uniform(2.0, 10.0, size=3) para esp). Qualquer mudança no
    CLI _build_models DEVE ser espelhada aqui para preservar a comparação
    apples-to-apples Numba CLI vs JAX batched.
    """
    rng = np.random.default_rng(seed)
    models = []
    for _ in range(n_models):
        rho_h = rng.uniform(1.0, 100.0, size=5).astype(np.float64)
        rho_v = rho_h * rng.uniform(1.0, 3.0, size=5).astype(np.float64)
        esp   = rng.uniform(2.0, 10.0, size=3).astype(np.float64)  # n=5 → 3 espessuras
        models.append({"rho_h": rho_h, "rho_v": rho_v, "esp": esp})
    return models


def run_benchmark_batched_scenario(scen_name: str, n_models: int, n_runs_hot: int = 4) -> dict:
    """Executa benchmark batched de um cenário com Run 1 cold isolado.

    Run 1 (cold): primeira chamada após warmup global; deve reusar XLA cache
    se warmup cobriu o batch size exato. Reportado separadamente.

    Runs 2-(n_runs_hot+1): hot-cache puro. statistics.median sobre estes
    é a métrica oficial do gate (C2 fix).

    Args:
        scen_name: identificador do cenário (A..H)
        n_models: número de modelos no batch
        n_runs_hot: runs hot-cache após Run 1 cold (default 4 → total 5)

    Returns:
        dict com run_1_cold_mod_h, hot_throughputs_mod_h, median_hot_mod_h,
        latency_ms_per_model_hot (CRIT-1 fix) e metadados.
    """
    scen = SCENARIOS_JAX[scen_name]
    n_pos = scen["n_pos"]
    positions_z = np.linspace(-5.0, 5.0, n_pos).astype(np.float64)  # match CLI benchmark.py:259
    models = _build_models(n_models, seed=SEED)

    rho_h_batch = np.stack([m["rho_h"] for m in models])  # (n_models, 5)
    rho_v_batch = np.stack([m["rho_v"] for m in models])  # (n_models, 5)
    esp_batch   = np.stack([m["esp"]   for m in models])  # (n_models, 3)

    kwargs = dict(
        frequencies_hz=list(scen["freqs"]),
        tr_spacings_m=list(scen["trs"]),
        dip_degs=list(scen["dips_jax"]),
    )

    # === Run 1 — COLD ===
    t0 = time.perf_counter()
    res = simulate_multi_jax_batched(rho_h_batch, rho_v_batch, esp_batch,
                                      positions_z, **kwargs)
    _ = res.H_tensor.shape  # H2 defensivo
    run_1_elapsed = time.perf_counter() - t0
    run_1_mod_h = (n_models / run_1_elapsed) * 3600.0
    print(f"    Run 1 (cold): {run_1_mod_h:>12,.0f} mod/h  ({run_1_elapsed:.3f}s)")

    # === Runs 2 a (n_runs_hot + 1) — HOT ===
    hot_throughputs = []
    for i in range(n_runs_hot):
        t0 = time.perf_counter()
        res = simulate_multi_jax_batched(rho_h_batch, rho_v_batch, esp_batch,
                                          positions_z, **kwargs)
        _ = res.H_tensor.shape  # H2 defensivo
        elapsed = time.perf_counter() - t0
        th = (n_models / elapsed) * 3600.0
        hot_throughputs.append(th)
        print(f"    Run {i+2}  (hot):  {th:>12,.0f} mod/h  ({elapsed:.3f}s)")

    median_hot = statistics.median(hot_throughputs)
    # CRIT-1 fix: ms POR MODELO = 3.6e6 ms/h / throughput em mod/h
    # (NÃO n_models / throughput * 3.6e6, que dá ms para TODOS os modelos)
    latency_ms_per_model_hot = 3_600_000.0 / median_hot

    return {
        "scenario": scen_name,
        "n_models": n_models,
        "n_pos": n_pos,
        "nf": len(scen["freqs"]),
        "nTR": len(scen["trs"]),
        "nAng": len(scen["dips_jax"]),
        "n_runs_hot": n_runs_hot,
        "run_1_cold_mod_h": run_1_mod_h,
        "run_1_cold_elapsed_s": run_1_elapsed,
        "hot_throughputs_mod_h": hot_throughputs,
        "median_hot_mod_h": median_hot,
        "min_hot_mod_h": min(hot_throughputs),
        "max_hot_mod_h": max(hot_throughputs),
        "ratio_cold_to_hot": run_1_mod_h / median_hot,
        "latency_ms_per_model_hot": latency_ms_per_model_hot,
        "numba_t4_local_mod_h": numba_t4_baselines.get(scen_name),
        "numba_baseline_historical_mod_h": scen["numba_baseline_historical_mod_h"],
    }


results_batched = {}
print("=" * 75)
print("BENCHMARK — simulate_multi_jax_batched (Sprint A1.5, v2.42)")
print("=" * 75)

for scen_name in SCENARIOS_JAX:
    n_m = N_MODELS_PER_SCENARIO[scen_name]
    scen = SCENARIOS_JAX[scen_name]
    n_combos = len(scen["trs"]) * len(scen["dips_jax"]) * len(scen["freqs"])
    print(f"\nCenário {scen_name} — n_pos={scen['n_pos']}, {n_combos} combos (nf×nTR×nAng), n_models={n_m}")
    try:
        results_batched[scen_name] = run_benchmark_batched_scenario(
            scen_name, n_models=n_m, n_runs_hot=N_RUNS_HOT,
        )
    except Exception as _exc:
        if "RESOURCE_EXHAUSTED" in str(_exc) or "Out of memory" in str(_exc):
            print(f"  → OOM com n_models={n_m} — cenário {scen_name} pulado")
            results_batched[scen_name] = {
                "scenario": scen_name, "n_models": n_m, "n_pos": scen["n_pos"],
                "nf": len(scen["freqs"]), "nTR": len(scen["trs"]), "nAng": len(scen["dips_jax"]),
                "n_runs_hot": N_RUNS_HOT, "run_1_cold_mod_h": None, "run_1_cold_elapsed_s": None,
                "hot_throughputs_mod_h": [], "median_hot_mod_h": None,
                "min_hot_mod_h": None, "max_hot_mod_h": None, "ratio_cold_to_hot": None,
                "latency_ms_per_model_hot": None, "numba_t4_local_mod_h": None,
                "oom": True,
            }
            continue
        else:
            raise
    r = results_batched[scen_name]
    nb_t4_local = r["numba_t4_local_mod_h"]
    if scen_name == "H":
        # M1: H comparison não é apples-to-apples (87.5° JAX vs 90° Numba)
        ratio_str = "  (Numba T4: 87.5° vs 90° — M1 doc)" if nb_t4_local else "  (Numba T4 indisponível)"
    else:
        ratio_str = f"  ({r['median_hot_mod_h']/nb_t4_local:.2f}× Numba T4)" if nb_t4_local else "  (Numba T4 indisponível)"
    print(f"  → Hot median: {r['median_hot_mod_h']:>12,.0f} mod/h{ratio_str}")
    print(f"    Cold/hot ratio: {r['ratio_cold_to_hot']:.2f}  (≈1.0 = warmup efetivo, >2× = warmup falhou)")
    print(f"    Latência: {r['latency_ms_per_model_hot']:.3f} ms/modelo")

print("\n✓ Benchmark batched concluído.")


BENCHMARK — simulate_multi_jax_batched (Sprint A1.5, v2.42)

Cenário A — n_pos=1, 1 combos (nf×nTR×nAng), n_models=50
    Run 1 (cold):    5,978,418 mod/h  (0.030s)
    Run 2  (hot):     6,706,242 mod/h  (0.027s)
    Run 3  (hot):     7,035,038 mod/h  (0.026s)
    Run 4  (hot):     7,113,373 mod/h  (0.025s)
    Run 5  (hot):     6,764,037 mod/h  (0.027s)
  → Hot median:    6,899,537 mod/h  (2.56× Numba T4)
    Cold/hot ratio: 0.87  (≈1.0 = warmup efetivo, >2× = warmup falhou)
    Latência: 0.522 ms/modelo

Cenário B — n_pos=100, 1 combos (nf×nTR×nAng), n_models=50
    Run 1 (cold):      234,304 mod/h  (0.768s)
    Run 2  (hot):       254,233 mod/h  (0.708s)
    Run 3  (hot):       259,087 mod/h  (0.695s)
    Run 4  (hot):       256,826 mod/h  (0.701s)
    Run 5  (hot):       258,432 mod/h  (0.697s)
  → Hot median:      257,629 mod/h  (2.86× Numba T4)
    Cold/hot ratio: 0.91  (≈1.0 = warmup efetivo, >2× = warmup falhou)
    Latência: 13.974 ms/modelo

Cenário C — n_pos=100, 4 combos (n

## § 7 — Tabela comparativa + Gates (Sprint O1)

**Duas comparações independentes:**

1. **Ganho O1 vs Baseline pré-O1** (`.claude/perf_baseline.json` → `jax_gpu_t4` ou `jax_gpu_a100`):
   - `medido_hot / baseline_pre_O1`
   - **Objetivo da Sprint O1**: razão > 1.0 indica ganho da otimização
   - Comparado contra `O1_GAIN_EXPECTATIONS` (configurado em §1)

2. **Gate Sprint A1.6 vs Numba T4 LOCAL** (medido via subprocess CLI no mesmo Colab):
   - `medido_hot / numba_t4_local`
   - **Critério herdado**: ≥1.5× em A, B, E (preserva gate A1.6)
   - H NÃO entra (dip 87.5° JAX vs 90° Numba)


In [ ]:
# ── Carregar baseline pré-O1 (jax_gpu_t4 ou jax_gpu_a100) ────────────────────
_HW_TO_SECTION = {"a100": "jax_gpu_a100", "t4": "jax_gpu_t4"}
BASELINE_SECTION = _HW_TO_SECTION.get(RUNTIME_LABEL, "jax_gpu_a100")
BASELINE_PATH = Path(REPO_DIR) / ".claude" / "perf_baseline.json"

_pre_o1_baseline = None
try:
    with BASELINE_PATH.open(encoding="utf-8") as _f:
        _pre_o1_baseline = json.load(_f).get(BASELINE_SECTION, {})
    print(f"Baseline pré-O1 carregada: seção {BASELINE_SECTION!r} ({len(_pre_o1_baseline)} entradas)")
except Exception as _e:
    print(f"⚠ Baseline pré-O1 indisponível: {_e}")
    _pre_o1_baseline = {}

print("=" * 140)
print(f"{'Cen':>3} | {'n_pos':>5} | {'nf×TR×Ang':>9} | "
      f"{'Numba T4 LOCAL':>14} | {'JAX cold':>11} | {'JAX hot (med)':>13} | "
      f"{'vs Numba':>9} | {'vs pré-O1':>10} | {'Expect O1':>9} | {'Status O1':>10} | {'C/H':>5}")
print("-" * 140)

GATE_PASS_SCENARIOS = ["A", "B", "E"]
gate_a16_results = {}
gate_o1_results = {}

for scen_name in SCENARIOS_JAX:
    scen   = SCENARIOS_JAX[scen_name]
    r      = results_batched[scen_name]
    nb_t4  = r["numba_t4_local_mod_h"]
    combos = f"{len(scen['freqs'])}×{len(scen['trs'])}×{len(scen['dips_jax'])}"

    cold_str = f"{r['run_1_cold_mod_h']:>10,.0f}" if r['run_1_cold_mod_h'] is not None else f"{' (falha)':>10}"
    hot_str  = f"{r['median_hot_mod_h']:>12,.0f}" if r['median_hot_mod_h'] is not None else f"{' (falha)':>12}"
    nb_str   = f"{nb_t4:>12,.0f}" if nb_t4 else f"{'(falha)':>12}"

    # Razão vs Numba T4
    if scen_name == "H" and nb_t4:
        ratio_t4_str = f"{'(dip≠)':>8}"  # 87.5° JAX vs 90° Numba
    else:
        ratio_t4_str = f"{r['median_hot_mod_h']/nb_t4:>7.2f}×" if nb_t4 else f"{'—':>8}"

    # Razão vs Baseline pré-O1
    pre_o1_hot = (_pre_o1_baseline.get(f"{scen_name}_hot", {}) or {}).get("throughput_mod_h")
    if pre_o1_hot:
        ratio_o1 = r["median_hot_mod_h"] / pre_o1_hot
        ratio_o1_str = f"{ratio_o1:>8.2f}×"
        expected = O1_GAIN_EXPECTATIONS.get(scen_name, 1.00)
        expect_str = f"{expected:>7.2f}×"
        # Status: ✓ se atingiu expectativa, ⚠ se neutro (sem regressão >5%), ✗ se regrediu
        if ratio_o1 >= expected:
            status_o1 = "✓ ATINGIU"
        elif ratio_o1 >= 0.95:
            status_o1 = "○ NEUTRO"
        else:
            status_o1 = "✗ REGREDIU"
        gate_o1_results[scen_name] = {
            "median_hot_mod_h": r["median_hot_mod_h"],
            "pre_o1_baseline_mod_h": pre_o1_hot,
            "ratio_vs_pre_o1": ratio_o1,
            "expected_gain": expected,
            "atingiu_expectativa": ratio_o1 >= expected,
            "sem_regressao": ratio_o1 >= 0.95,
        }
    else:
        ratio_o1_str = f"{'—':>9}"
        expect_str = f"{'—':>8}"
        status_o1 = "?"

    ch_str = f"{r['ratio_cold_to_hot']:>5.2f}" if r['ratio_cold_to_hot'] is not None else f"{'—':>5}"

    print(f"{scen_name:>3} | {scen['n_pos']:>5} | {combos:>9} | "
          f"{nb_str:>14} m/h | {cold_str} m/h | {hot_str} m/h | "
          f"{ratio_t4_str} | {ratio_o1_str} | {expect_str} | {status_o1:>10} | {ch_str}")

    if scen_name in GATE_PASS_SCENARIOS and nb_t4:
        ratio_a16 = r["median_hot_mod_h"] / nb_t4
        gate_a16_results[scen_name] = {
            "median_hot_mod_h": r["median_hot_mod_h"],
            "numba_t4_local_mod_h": nb_t4,
            "ratio_hot_vs_t4": ratio_a16,
            "pass": ratio_a16 >= 1.5,
        }

print("=" * 140)
print("  vs Numba   = JAX_hot / Numba_T4_LOCAL (gate Sprint A1.6: ≥1.5×)")
print("  vs pré-O1  = JAX_hot / baseline_jax_pre_O1 (objetivo Sprint O1: >1.0 indica ganho real)")
print("  Expect O1  = ganho mínimo esperado por cenário (cell-03 O1_GAIN_EXPECTATIONS)")
print("  Status O1  = ✓ ATINGIU expectativa | ○ NEUTRO (±5%) | ✗ REGREDIU (<0.95×)")
print("  C/H        = ratio cold/hot (~1.0 = warmup efetivo)")
print("  H: JAX usa dip 87.5° (cap 89°), Numba usa 90° — razão vs Numba não comparável")

# ─────────────────────────────────────────────────────────────────────────────
# Gate 1 — Sprint A1.6 (≥1.5× Numba T4 LOCAL em A/B/E)
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 90)
print("GATE A1.6 (HERDADO) — ≥ 1.5× Numba T4 LOCAL em A, B, E")
print("=" * 90)
all_pass_a16 = True
a16_evaluable = False
for scen_name in GATE_PASS_SCENARIOS:
    if scen_name not in gate_a16_results:
        print(f"  Cenário {scen_name}: Numba T4 LOCAL indisponível → não avaliável")
        continue
    g = gate_a16_results[scen_name]
    a16_evaluable = True
    status = "✓ PASS" if g["pass"] else "✗ FAIL"
    print(f"  Cenário {scen_name}: {g['median_hot_mod_h']:>12,.0f} mod/h ({g['ratio_hot_vs_t4']:.2f}× Numba T4) → {status}")
    if not g["pass"]:
        all_pass_a16 = False

GATE_A16_PASS = a16_evaluable and all_pass_a16
print(f"\n  Gate A1.6 global: {'✓ APROVADO' if GATE_A16_PASS else '✗ REPROVADO' if a16_evaluable else '? NÃO AVALIÁVEL'}")

# ─────────────────────────────────────────────────────────────────────────────
# Gate 2 — Sprint O1 (medido > 0.95× baseline pré-O1 em TODOS os cenários
#                     com ganho ≥ expectativa em pelo menos 1 cenário)
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 90)
print("GATE O1 — Ganhos vs baseline pré-O1 (anti-regressão + ≥1 cenário atinge expectativa)")
print("=" * 90)
sem_regressao = []
atingiu_expectativa = []
for scen_name, g in gate_o1_results.items():
    sem_regressao.append(g["sem_regressao"])
    atingiu_expectativa.append(g["atingiu_expectativa"])
    if not g["sem_regressao"]:
        print(f"  ⚠ {scen_name}: REGRESSÃO {g['ratio_vs_pre_o1']:.2f}× (baseline={g['pre_o1_baseline_mod_h']:,.0f}, medido={g['median_hot_mod_h']:,.0f})")

n_sem_reg = sum(sem_regressao)
n_atingiu = sum(atingiu_expectativa)
n_total = len(gate_o1_results)
print(f"\n  Sem regressão (≥0.95×): {n_sem_reg}/{n_total} cenários")
print(f"  Atingiu expectativa O1: {n_atingiu}/{n_total} cenários")

# O1 PASS: nenhuma regressão E pelo menos 1 cenário atingiu expectativa
GATE_O1_PASS = (n_sem_reg == n_total) and (n_atingiu >= 1)
if GATE_O1_PASS:
    print(f"\n  Gate O1: ✓ APROVADO — sem regressões + {n_atingiu} cenário(s) atingiu(ram) expectativa")
elif n_sem_reg == n_total:
    print(f"\n  Gate O1: ○ NEUTRO — sem regressões, mas nenhum cenário atingiu expectativa de ganho")
else:
    print(f"\n  Gate O1: ✗ REPROVADO — {n_total - n_sem_reg} cenário(s) com regressão >5%")


## § 8 — Exportar Resultados para Drive


In [ ]:
# RUNTIME_LABEL já detectado em §2 setup (cell-05). Não re-detectar.
print(f"Hardware detectado: {RUNTIME_LABEL}")

ts = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
os.makedirs(OUTPUT_DRIVE_DIR, exist_ok=True)
out_fname = f"sprint_o0_o1_full_benchmark_{RUNTIME_LABEL}_{ts}.json"
out_path  = os.path.join(OUTPUT_DRIVE_DIR, out_fname)

output = {
    "sprint":          "O0+O1 (full benchmark — 8 scenarios)",
    "template":        "validate_jax_gpu_sprinto0_o1 (cópia adaptada de v240)",
    "git_branch":      GIT_BRANCH,
    "git_commit":      GIT_COMMIT,
    "runtime":         RUNTIME_LABEL,
    "baseline_section": BASELINE_SECTION,
    "timestamp_utc":   ts,
    "jax_version":     jax.__version__,
    "config": {
        "n_models_per_scenario": N_MODELS_PER_SCENARIO,
        "n_runs_hot": N_RUNS_HOT,
        "seed": SEED,
        "paridade_tol": PARIDADE_TOL,
        "jax_enable_x64": True,
        "api": "simulate_multi_jax_batched",
        "positions_z_range": "[-5.0, 5.0]",
        "numba_baseline_workers": NUMBA_BASELINE_WORKERS,
        "numba_baseline_threads": NUMBA_BASELINE_THREADS,
        "numba_cli_timeout_s": NUMBA_CLI_TIMEOUT_S,
        "o1_gain_expectations": O1_GAIN_EXPECTATIONS,
    },
    "env_vars": {
        "JAX_ENABLE_X64":               os.environ.get("JAX_ENABLE_X64"),
        "JAX_PLATFORMS":                os.environ.get("JAX_PLATFORMS"),
        "JAX_COMPILATION_CACHE_DIR":    os.environ.get("JAX_COMPILATION_CACHE_DIR"),
        "XLA_FLAGS":                    os.environ.get("XLA_FLAGS"),
        "XLA_PYTHON_CLIENT_PREALLOCATE": os.environ.get("XLA_PYTHON_CLIENT_PREALLOCATE"),
        "XLA_PYTHON_CLIENT_MEM_FRACTION": os.environ.get("XLA_PYTHON_CLIENT_MEM_FRACTION"),
    },
    "pytest_summary": _pytest_report["summary"],
    "pytest_duration_sec": _pytest_report.get("duration", 0.0),
    "benchmark_batched": results_batched,
    "numba_t4_local_baselines": numba_t4_baselines,
    "quick_wins_details": [
        {"label": label, "ok": ok, "detail": str(detail)}
        for label, ok, detail in qw_checks
    ],
    "quick_wins_n_pass": _qw_n_pass,
    "quick_wins_n_total": len(qw_checks),
    "quick_wins_all_pass": QW_ALL_PASS,
    "gate_a16": {
        "criterion": "≥1.5× Numba T4 LOCAL em A, B, E",
        "results": gate_a16_results,
        "pass": GATE_A16_PASS,
    },
    "gate_o1": {
        "criterion": "sem regressão >5% em qualquer cenário E ≥1 cenário atingiu expectativa",
        "results": gate_o1_results,
        "pass": GATE_O1_PASS,
        "expectations": O1_GAIN_EXPECTATIONS,
    },
    "pre_o1_baseline_section": BASELINE_SECTION,
    "pre_o1_baseline_meta": _pre_o1_baseline.get("_meta", {}) if _pre_o1_baseline else {},
    "h_scenario_note": "dip_degs max=87.5° (JAX validador cap 89°) vs 90° Numba CLI (M1 doc, gate A1.6 só A/B/E)",
}

with open(out_path, "w") as _f:
    json.dump(output, _f, indent=2, default=str)

print(f"✓ Resultados exportados: {out_path}")
print(f"  Tamanho: {os.path.getsize(out_path) / 1024:.1f} KB")


## § 9 — Resumo Final


In [ ]:
_pytest_sum = _pytest_report["summary"]

print("=" * 90)
print("SPRINT O0/O1 — BENCHMARK JAX GPU COMPLETO (8 CENÁRIOS) CONCLUÍDA")
print(f"Branch:   {GIT_BRANCH}")
print(f"Commit:   {GIT_COMMIT[:12]}")
print(f"Hardware: {RUNTIME_LABEL.upper()}  (baseline pré-O1: {BASELINE_SECTION})")
print("=" * 90)

print(f"\nParidade Fortran <1e-12 (pytest):")
print(f"  PASS:    {_pytest_sum.get('passed', 0)}")
print(f"  FAIL:    {_pytest_sum.get('failed', 0)}")
print(f"  SKIPPED: {_pytest_sum.get('skipped', 0)}")

print(f"\nQuick Wins Sprint O1: {_qw_n_pass}/{len(qw_checks)} {'✓' if QW_ALL_PASS else '⚠'}")

print("\nThroughput por cenário (mediana hot) — comparações duplas:")
print(f"{'Cen':>3} | {'hot mod/h':>14} | {'vs Numba T4':>11} | {'vs pré-O1':>10} | {'Expect O1':>9} | {'Status O1':>10}")
print("-" * 80)
for scen_name in SCENARIOS_JAX:
    r = results_batched[scen_name]
    nb_t4 = r["numba_t4_local_mod_h"]
    hot = r["median_hot_mod_h"]
    hot_str = f"{hot:>12,.0f}" if hot is not None else f"{'(falha)':>12}"

    if scen_name == "H":
        ratio_t4_str = f"{'(dip≠)':>9}"
    elif nb_t4 and hot:
        ratio_t4_str = f"{hot/nb_t4:>8.2f}×"
    else:
        ratio_t4_str = f"{'—':>9}"

    o1 = gate_o1_results.get(scen_name)
    if o1:
        ratio_o1_str = f"{o1['ratio_vs_pre_o1']:>8.2f}×"
        expect_str = f"{o1['expected_gain']:>7.2f}×"
        if o1["atingiu_expectativa"]:
            status_o1 = "✓ ATINGIU"
        elif o1["sem_regressao"]:
            status_o1 = "○ NEUTRO"
        else:
            status_o1 = "✗ REGREDIU"
    else:
        ratio_o1_str = f"{'—':>9}"
        expect_str = f"{'—':>8}"
        status_o1 = "?"
    print(f"{scen_name:>3} | {hot_str} m/h | {ratio_t4_str} | {ratio_o1_str} | {expect_str} | {status_o1:>10}")

print("\n" + "=" * 90)
print("GATES FINAIS")
print("=" * 90)
print(f"  Gate A1.6 (≥1.5× Numba T4 em A,B,E): {'✓ APROVADO' if GATE_A16_PASS else '✗ REPROVADO'}")
print(f"  Gate O1   (sem regressão + ≥1 expect): {'✓ APROVADO' if GATE_O1_PASS else '✗ REPROVADO'}")
print(f"  Quick Wins ativos: {_qw_n_pass}/{len(qw_checks)} {'✓' if QW_ALL_PASS else '⚠'}")
print(f"  Paridade Fortran: {_pytest_sum.get('passed', 0)} PASS / {_pytest_sum.get('failed', 0)} FAIL")

print(f"\nResultados salvos em: {out_path}")
print("=" * 90)

print("""
PRÓXIMOS PASSOS (após reportar JSON para nova sessão Claude):
  1. Claude analisa ganhos por cenário vs expectativas O1
  2. Identifica quais Quick Wins entregaram valor real (C/D/G mais sensíveis)
  3. Decide Sprint O2: focar nos cenários onde O1 não atingiu expectativa
  4. Atualiza docs/PERFORMANCE_BASELINE.md + ROADMAP §0
  5. Considera UPDATE_BASELINE=True em validate_sprint_o0_o1_gpu.ipynb se ganhos
     forem consistentes (>5% em ≥2 cenários)
""")


## § A — Sprint O0 Profiling Roofline (A100)

**Objetivo**: identificar se o bottleneck do simulador JAX GPU é
**compute-bound**, **memory-bound** ou **latency-bound**, para guiar
as próximas sprints de otimização (O1–O4).

**Como usar**:
1. Execute esta célula em Colab Pro+ A100
2. Download do trace `.json.gz` do diretório `/tmp/jax_trace_e/`
3. Abra em https://ui.perfetto.dev/ → carregue o JSON
4. Inspecione: tempo gasto em compute vs HBM vs kernel launches

**Métricas a coletar e documentar em `docs/reports/v2.43_jax_profile_baseline_a100.md`**:
- VRAM peak vs limite (40 GB A100)
- Decomposição de tempo: compute / kernel launch / sync / memcpy
- Distribuição dos 6 casos `lax.switch` (skewed ou uniforme?)
- HBM bandwidth saturation (% do pico 1555 GB/s)

Esta célula NÃO afeta o benchmark — é diagnóstica.


In [ ]:
# Sprint O0 (T0.2) — Profile roofline JAX GPU em cenário E (worst case dominante)
# Output: /tmp/jax_trace_e/<timestamp>/ → arquivos plugins/profile/*.json.gz
# Pós-execução: baixe e abra em https://ui.perfetto.dev/

import os
import time
from pathlib import Path

import numpy as np
import jax
from jax import profiler

from geosteering_ai.simulation._jax.multi_forward import (
    simulate_multi_jax_batched,
)

# ── Configuração do cenário E (mais informativo: dominante em GPU usage) ──
N_MODELS_PROFILE = 50      # mesmo que baseline A100
N_POS_PROFILE = 600        # cenário E
N_FREQS_PROFILE = 1
N_TR_PROFILE = 1
N_ANG_PROFILE = 1

TRACE_DIR = Path("/tmp/jax_trace_e")
TRACE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Trace dir: {TRACE_DIR}")
print(f"JAX backend: {jax.default_backend()}, devices: {jax.devices()}")

# ── Modelo sintético reproduzível ─────────────────────────────────────────
rng = np.random.default_rng(42)
rho_h_batch = rng.uniform(1.0, 100.0, size=(N_MODELS_PROFILE, 3)).astype(np.float64)
rho_v_batch = rho_h_batch.copy()
esp_batch = np.full((N_MODELS_PROFILE, 1), 5.0, dtype=np.float64)
positions_z = np.linspace(-5.0, 5.0, N_POS_PROFILE).astype(np.float64)

kwargs = dict(
    frequencies_hz=[20000.0],
    tr_spacings_m=[1.0],
    dip_degs=[0.0],
)

# ── Warmup (descarta cold-start XLA compile do trace) ─────────────────────
print("Warmup (descarta cold-start)...")
_ = simulate_multi_jax_batched(
    rho_h_batch, rho_v_batch, esp_batch, positions_z, **kwargs
).H_tensor.shape

# ── Trace de 1 batch hot ──────────────────────────────────────────────────
print("Tracing batch hot (cenário E)...")
with profiler.trace(str(TRACE_DIR), create_perfetto_link=False):
    t0 = time.perf_counter()
    result = simulate_multi_jax_batched(
        rho_h_batch, rho_v_batch, esp_batch, positions_z, **kwargs
    )
    _ = result.H_tensor.shape  # força sync GPU→CPU
    elapsed = time.perf_counter() - t0

throughput = N_MODELS_PROFILE / elapsed * 3600.0
print(f"\n✓ Trace concluído:")
print(f"  Tempo: {elapsed*1000:.2f} ms")
print(f"  Throughput: {throughput:,.0f} mod/h")
print(f"  Trace files: {list(TRACE_DIR.rglob('*.json.gz'))[:3]}")
print(f"\nProximos passos:")
print(f"  1. Download da pasta {TRACE_DIR}")
print(f"  2. Abrir .json.gz em https://ui.perfetto.dev/")
print(f"  3. Documentar findings em docs/reports/v2.43_jax_profile_baseline_a100.md")
